In [196]:
import pandas as pd
import numpy as np

### Function to read the .txt files into a DataFrame

In [197]:

df_list = []
file_locs = ['./scraped_data/data.txt','./scraped_data/cardekho.txt']
for file in file_locs:
    with open(file, 'r', encoding='utf-8') as rf:
        for line in rf:
            inp = eval(line)
            main_dict = {}

            for key, value in inp.items():
                if(key == 'info'):
                    for keyinfo,valueinfo in inp[key].items():
                        main_dict[keyinfo] = valueinfo   
                else:
                        main_dict[key] = value
            df_list.append(main_dict)

df = pd.DataFrame(df_list)

# Basic Overview of the Data

In [198]:
df.head()

,model,price,price_score,plate,Reg. year,Fuel,KM driven,Transmission,Engine capacity,Ownership,Make year,Spare key,Reg number,Insurance,Insurance type
0,2020 Jeep Compass LONGITUDE PLUS 1.4 PETROL AT,₹8.50 lakh,STEAL DEAL according to our Price Indicator.,DL-10,Jul 2020,Petrol,"116,562 km",Automatic,1368cc,1st,Feb 2020,No,******8927,Nov 2026,3rd Party
1,2016 Mahindra XUV500 W6 AT 1.99,₹4.51 lakh,FAIR PRICE according to our Price Indicator.,DL-10,Aug 2016,Diesel,"238,933 km",Automatic,1997cc,1st,Jun 2016,No,******6357,NaN,NaN
2,2012 Hyundai Eon D-LITE,₹0.84 lakh,STEAL DEAL according to our Price Indicator.,UP-15,Jan 2013,CNG,"94,507 km",Manual,814cc,2nd,Jan 2012,No,******2716,Nov 2026,3rd Party
3,2013 Hyundai i20 SPORTZ 1.2,₹1.81 lakh,STEAL DEAL according to our Price Indicator.,DL-10,Mar 2013,CNG,"120,121 km",Manual,1197cc,2nd,Feb 2013,No,******7982,Dec 2026,3rd Party
4,2014 Honda City 1.5L I-VTEC SV,₹3.07 lakh,STEAL DEAL according to our Price Indicator.,DL-4C,May 2014,Petrol,"77,884 km",Manual,1497cc,2nd,Apr 2014,No,******9657,NaN,NaN


In [199]:
print(f"There are {df.duplicated().value_counts().iloc[1]} duplicated rows.")

There are 2197 duplicated rows.


# Modifying Datatypes :

In [201]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14595 entries, 0 to 14594
Data columns (total 15 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   model            14595 non-null  object
 1   price            14595 non-null  object
 2   price_score      14295 non-null  object
 3   plate            14595 non-null  object
 4   Reg. year        14595 non-null  object
 5   Fuel             14595 non-null  object
 6   KM driven        14595 non-null  object
 7   Transmission     14595 non-null  object
 8   Engine capacity  14302 non-null  object
 9   Ownership        14595 non-null  object
 10  Make year        14595 non-null  object
 11  Spare key        9145 non-null   object
 12  Reg number       9145 non-null   object
 13  Insurance        4105 non-null   object
 14  Insurance type   9555 non-null   object
dtypes: object(15)
memory usage: 1.7+ MB


In [202]:
df.head()

,model,price,price_score,plate,Reg. year,Fuel,KM driven,Transmission,Engine capacity,Ownership,Make year,Spare key,Reg number,Insurance,Insurance type
0,2020 Jeep Compass LONGITUDE PLUS 1.4 PETROL AT,₹8.50 lakh,STEAL DEAL according to our Price Indicator.,DL-10,Jul 2020,Petrol,"116,562 km",Automatic,1368cc,1st,Feb 2020,No,******8927,Nov 2026,3rd Party
1,2016 Mahindra XUV500 W6 AT 1.99,₹4.51 lakh,FAIR PRICE according to our Price Indicator.,DL-10,Aug 2016,Diesel,"238,933 km",Automatic,1997cc,1st,Jun 2016,No,******6357,NaN,NaN
2,2012 Hyundai Eon D-LITE,₹0.84 lakh,STEAL DEAL according to our Price Indicator.,UP-15,Jan 2013,CNG,"94,507 km",Manual,814cc,2nd,Jan 2012,No,******2716,Nov 2026,3rd Party
3,2013 Hyundai i20 SPORTZ 1.2,₹1.81 lakh,STEAL DEAL according to our Price Indicator.,DL-10,Mar 2013,CNG,"120,121 km",Manual,1197cc,2nd,Feb 2013,No,******7982,Dec 2026,3rd Party
4,2014 Honda City 1.5L I-VTEC SV,₹3.07 lakh,STEAL DEAL according to our Price Indicator.,DL-4C,May 2014,Petrol,"77,884 km",Manual,1497cc,2nd,Apr 2014,No,******9657,NaN,NaN


- `model` to only include the main name of the model. 
- `price`, `KM driven` and `Engine capacity` to be converted into a numeric format
- `price_score` to be trimmed down.
- `plate` to be standardised.
- `Reg year` and `Make year` to be converted into datetime objects.


In [203]:
# creating a copy:
data = df.copy()

In [204]:
# trimming model column :
data['model'] = data['model'].str.replace('\n',' ') # cleaning up the newlines
data['model'] = data['model'].str.split(' ').map(lambda x: x[1]+' '+x[2] if x[2] != 'Suzuki' else x[1]+' '+x[2]+' '+x[3])
data['model'].value_counts()[:10] # FIX MAruti SUZUKI

model
Honda City       749
Hyundai Creta    735
Maruti Swift     471
Honda Amaze      464
Maruti Wagon     406
Hyundai Grand    399
Hyundai i10      380
Ford Ecosport    326
Tata NEXON       320
Maruti Baleno    311
Name: count, dtype: int64

In [205]:
# replacing price column :
def price_conversion(x):
    x = x.lower().replace('₹','')

    if(x.endswith('lakh')):
        x = x.replace('lakh','')
        x = float(x)*100000

    elif(x.endswith('thousand')):
        x = x.replace('thousand','')
        x = float(x)*1000
    
    return(x)

data['price'] = data['price'].map(price_conversion)

In [206]:
# converting KM driven and engine capacity column:
data['KM driven'] = data['KM driven'].str.replace(r'[A-Za-z,\s]+', '', regex=True).astype('int')
data['Engine capacity'] = data['Engine capacity'].str.strip().str.replace('cc','').astype('Int32')

In [207]:
# trimming price_score column :
data['price_score'] = data['price_score'].replace('NA', np.nan)
data.loc[data['price_score'].notna(), 'price_score'] = data.loc[data['price_score'].notna(), 'price_score'].str.split(' ').map(lambda x: x[0]+' '+x[1])

In [208]:
# converting Reg. year and Make year
data['Reg. year'] = pd.to_datetime(data['Reg. year']).dt.year
data['Make year'] = pd.to_datetime(data['Make year']).dt.year

C:\Users\sisfi\AppData\Local\Temp\ipykernel_8604\4109392050.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data['Reg. year'] = pd.to_datetime(data['Reg. year']).dt.year
C:\Users\sisfi\AppData\Local\Temp\ipykernel_8604\4109392050.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  data['Make year'] = pd.to_datetime(data['Make year']).dt.year


In [215]:
# map plates :
data['plate'].value_counts()
data['plate'] = data['plate'].replace(r'[0-9-]+','', regex=True)
mappings = {'DL':'New Delhi', 'DLC':'New Delhi', 'UP':'Uttar Pradesh', 'Noida':'Uttar Pradesh', 
            'HR':'Haryana', 'Gurgaon':'Haryana', 'Faridabad':'Haryana', 'Ghaziabad':'Uttar Pradesh',
            'ballabhgarh': 'Haryana', 'UK':'Uttarakhand', 'Lucknow':'Uttar Pradesh', 'palwal':'Haryana'
            ,'BH':'Bharat','RJ':'Rajasthan','PB':'Punjab'}
print(data['plate'].replace(mappings).value_counts()[:100])

plate
New Delhi        7339
Haryana          3987
Uttar Pradesh    2300
Chandigarh         54
Uttarakhand        53
                 ... 
Mewat               2
KL                  1
GA                  1
DLS                 1
DLD                 1
Name: count, Length: 100, dtype: int64


# Examining Nulls :

In [216]:
df.isna().sum()

model                 0
price                 0
price_score         300
plate                 0
Reg. year             0
Fuel                  0
KM driven             0
Transmission          0
Engine capacity     293
Ownership             0
Make year             0
Spare key          5450
Reg number         5450
Insurance          5040
Insurance type     5040
dtype: int64

- `Spare Key` and `Reg Number` both have same number of null values, because they have been sourced from CarDekho, which does not provide this info.
- `price_score` field will have more nulls than depicted, because this info is not given in the CarDekho records either.
- `Engine capacity` nulls can be interpolated from other cars of the same model.
- `Insurance type` has less number of nulls than `Insurance`, because CarDekho data doesn't provide data for the 'Insurance' column.

In [ ]:
# Replacing Insurance type nulls w/ unknown:


In [223]:
car_grpby = data.groupby(['model','Make year'])
engine_grps = car_grpby['Engine capacity'].mean()
engine_grps

model          Make year
Audi A3        2017          1968.0
Audi A4        2014          1798.0
               2017         1538.25
               2022          1998.0
               2024          1984.0
                             ...   
Volvo XC60     2020          1969.0
               2022          1969.0
Volvo XC90     2020          1969.0
               2025          1969.0
gateway Error  2015          1197.0
Name: Engine capacity, Length: 1334, dtype: Float64